In [2]:
# Import pandas for data loading and DataFrame operations
import pandas as pd
# Import os for checking file paths and file existence
import os

In [3]:
# Define the path of the original CSV dataset
file_path = "export.csv"
# Check whether the dataset file exists at the given path
print("File exists:", os.path.exists(file_path))

File exists: True


In [4]:
# Load only the first 10,000 rows of the large dataset
# to avoid loading the complete 14.5 GB file into memory
df_sample = pd.read_csv(
    file_path,
    nrows=10_000
)
# Confirm that the sample dataset was loaded successfully
print("Sample loaded successfully!")
# Display the number of rows in the sample
print("Rows:", len(df_sample))

# Display the number of columns in the sample
print("Columns:", len(df_sample.columns))

Sample loaded successfully!
Rows: 10000
Columns: 44


In [5]:
# Display all column names available in the sample dataset
print(df_sample.columns.tolist())

['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location']


In [6]:
# Select only the columns that are relevant to the
# Urban Service Resolution Intelligence Platform project
selected_columns = [
    "Created Date",
    "Closed Date",
    "Agency",
    "Agency Name",
    "Problem (formerly Complaint Type)",
    "Problem Detail (formerly Descriptor)",
    "Location Type",
    "Incident Zip",
    "City",
    "Borough",
    "Community Board",
    "Council District",
    "Police Precinct",
    "Open Data Channel Type",
    "Latitude",
    "Longitude",
    "Status",
    "Due Date"
]
# Create a new DataFrame containing only the selected project-relevant columns
df_project = df_sample[selected_columns].copy()
# Create a new DataFrame containing only the selected project-relevant columns
print("Selected rows:", len(df_project))

# Display the number of selected columns
print("Selected columns:", len(df_project.columns))

# Display the names of the selected columns
print("\nColumns being used:")
print(df_project.columns.tolist())

Selected rows: 10000
Selected columns: 18

Columns being used:
['Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Location Type', 'Incident Zip', 'City', 'Borough', 'Community Board', 'Council District', 'Police Precinct', 'Open Data Channel Type', 'Latitude', 'Longitude', 'Status', 'Due Date']


In [7]:
# Display the current working directory of the Jupyter Notebook
print("Current folder:")
print(os.getcwd())

# Display the complete absolute path of the CSV dataset
print("\nCSV full path:")
print(os.path.abspath("export.csv"))

Current folder:
C:\Users\kumar\Urban

CSV full path:
C:\Users\kumar\Urban\export.csv


In [8]:
#It is check of dataset structure
df_project.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Created Date                          10000 non-null  object 
 1   Closed Date                           3906 non-null   object 
 2   Agency                                10000 non-null  object 
 3   Agency Name                           10000 non-null  object 
 4   Problem (formerly Complaint Type)     10000 non-null  object 
 5   Problem Detail (formerly Descriptor)  9821 non-null   object 
 6   Location Type                         8925 non-null   object 
 7   Incident Zip                          9900 non-null   float64
 8   City                                  9318 non-null   object 
 9   Borough                               10000 non-null  object 
 10  Community Board                       10000 non-null  object 
 11  Council District

In [9]:
#How many data missing in each column
missing = df_project.isnull().sum() 

missing = missing[missing > 0].sort_values(ascending=False)

print("Missing values:")
print(missing)

Missing values:
Due Date                                9974
Closed Date                             6094
Location Type                           1075
City                                     682
Council District                         262
Latitude                                 211
Longitude                                211
Problem Detail (formerly Descriptor)     179
Incident Zip                             100
dtype: int64


In [10]:
# Convert date columns to datetime format
df_project["Created Date"] = pd.to_datetime(
    df_project["Created Date"],
    errors="coerce"
)

df_project["Closed Date"] = pd.to_datetime(
    df_project["Closed Date"],
    errors="coerce"
)

# Calculate resolution time in days
df_project["Resolution Days"] = (
    df_project["Closed Date"] - df_project["Created Date"]
).dt.total_seconds() / (24 * 60 * 60)

# Check the new target
print("Resolution Days created successfully!")
print(df_project["Resolution Days"].describe())

Resolution Days created successfully!
count    3906.000000
mean        0.076271
std         0.100834
min        -0.000266
25%         0.013273
50%         0.038449
75%         0.096508
max         0.704630
Name: Resolution Days, dtype: float64


In [11]:
# Check the date range and invalid resolution times

print("Created Date range:")
print(df_project["Created Date"].min(), "to", df_project["Created Date"].max())

print("\nClosed Date range:")
print(df_project["Closed Date"].min(), "to", df_project["Closed Date"].max())

print("\nMissing Resolution Days:", df_project["Resolution Days"].isna().sum())

print("Negative Resolution Days:",
      (df_project["Resolution Days"] < 0).sum())

print("\nNegative resolution examples:")
print(
    df_project.loc[
        df_project["Resolution Days"] < 0,
        ["Created Date", "Closed Date", "Resolution Days"]
    ].head()
)

Created Date range:
2026-08-20 07:38:40 to 2026-08-21 02:05:14

Closed Date range:
2026-08-20 07:41:35 to 2026-08-21 03:17:00

Missing Resolution Days: 6094
Negative Resolution Days: 5

Negative resolution examples:
            Created Date         Closed Date  Resolution Days
1997 2026-08-20 20:24:11 2026-08-20 20:24:00        -0.000127
3315 2026-08-20 18:35:23 2026-08-20 18:35:00        -0.000266
3573 2026-08-20 18:08:04 2026-08-20 18:08:00        -0.000046
5052 2026-08-20 15:15:01 2026-08-20 15:15:00        -0.000012
9329 2026-08-20 08:56:09 2026-08-20 08:56:00        -0.000104


In [12]:
# Remove invalid negative resolution times
df_project.loc[
    df_project["Resolution Days"] < 0,
    "Resolution Days"
] = None

print("Negative Resolution Days:",
      (df_project["Resolution Days"] < 0).sum())

print("Missing Resolution Days:",
      df_project["Resolution Days"].isna().sum())

Negative Resolution Days: 0
Missing Resolution Days: 6099


In [13]:
# Keep only rows where the target is available
df_model = df_project.dropna(
    subset=["Resolution Days"]
).copy()

print("Model rows:", len(df_model))
print("Model columns:", len(df_model.columns))

Model rows: 3901
Model columns: 19


In [14]:
# Step 3: Separate input features (X) and target (y)

# Resolution Days is our target variable
y = df_model["Resolution Days"]

# Remove target and columns that should not be used for prediction
X = df_model.drop(
    columns=["Resolution Days", "Closed Date", "Due Date"]
).copy()

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

print("\nFeatures used for model:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features (X) shape: (3901, 16)
Target (y) shape: (3901,)

Features used for model:
['Created Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Location Type', 'Incident Zip', 'City', 'Borough', 'Community Board', 'Council District', 'Police Precinct', 'Open Data Channel Type', 'Latitude', 'Longitude', 'Status']

Target:
Resolution Days


In [15]:
# Step 4: Feature Analysis

# Check numerical features
print("Numerical Features:")
print(X.select_dtypes(include=["int64", "float64"]).columns.tolist())

# Check categorical features
print("\nCategorical Features:")
print(X.select_dtypes(include=["object"]).columns.tolist())

# Check number of unique values
print("\nUnique Values:")
print(X.nunique().sort_values(ascending=False))

# Check missing values in features
print("\nMissing Values in Features:")
print(X.isnull().sum().sort_values(ascending=False))

# Check target distribution
print("\nTarget (Resolution Days) Statistics:")
print(y.describe())

Numerical Features:
['Incident Zip', 'Council District', 'Latitude', 'Longitude']

Categorical Features:
['Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Location Type', 'City', 'Borough', 'Community Board', 'Police Precinct', 'Open Data Channel Type', 'Status']

Unique Values:
Created Date                            3656
Longitude                               3047
Latitude                                3047
Problem Detail (formerly Descriptor)     181
Incident Zip                             180
Police Precinct                           78
Community Board                           68
Problem (formerly Complaint Type)         68
Council District                          51
City                                      44
Location Type                             32
Agency Name                               11
Agency                                    11
Borough                                    6
Open Data Channel Type              

In [16]:
# Step 5: Feature preprocessing

# Convert Created Date into datetime format
X["Created Date"] = pd.to_datetime(X["Created Date"], errors="coerce")

# Create useful time-based features
X["Created_Year"] = X["Created Date"].dt.year
X["Created_Month"] = X["Created Date"].dt.month
X["Created_Day"] = X["Created Date"].dt.day
X["Created_DayOfWeek"] = X["Created Date"].dt.dayofweek

# Remove the original date column
X = X.drop(columns=["Created Date"])

print("Preprocessing started successfully!")
print("\nFeature shape:", X.shape)

print("\nMissing values:")
print(X.isnull().sum().sort_values(ascending=False))

Preprocessing started successfully!

Feature shape: (3901, 19)

Missing values:
Location Type                           556
City                                    279
Problem Detail (formerly Descriptor)    159
Council District                         73
Longitude                                54
Latitude                                 54
Incident Zip                             40
Problem (formerly Complaint Type)         0
Agency Name                               0
Agency                                    0
Borough                                   0
Police Precinct                           0
Community Board                           0
Open Data Channel Type                    0
Status                                    0
Created_Year                              0
Created_Month                             0
Created_Day                               0
Created_DayOfWeek                         0
dtype: int64


In [17]:
# Step 5: Handle missing values

# Fill missing categorical values with "Unknown"
categorical_cols = X.select_dtypes(include=["object"]).columns

X[categorical_cols] = X[categorical_cols].fillna("Unknown")

# Fill missing numerical values with median
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

X[numerical_cols] = X[numerical_cols].fillna(
    X[numerical_cols].median()
)

print("Missing values handled successfully!")

print("\nRemaining missing values:")
print(X.isnull().sum().sort_values(ascending=False))

Missing values handled successfully!

Remaining missing values:
Agency                                  0
Agency Name                             0
Problem (formerly Complaint Type)       0
Problem Detail (formerly Descriptor)    0
Location Type                           0
Incident Zip                            0
City                                    0
Borough                                 0
Community Board                         0
Council District                        0
Police Precinct                         0
Open Data Channel Type                  0
Latitude                                0
Longitude                               0
Status                                  0
Created_Year                            0
Created_Month                           0
Created_Day                             0
Created_DayOfWeek                       0
dtype: int64


In [18]:
# Step 6: Prepare numerical and categorical features

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical columns
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("\nPreprocessor created successfully!")

Numerical features:
['Incident Zip', 'Council District', 'Latitude', 'Longitude']

Categorical features:
['Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Location Type', 'City', 'Borough', 'Community Board', 'Police Precinct', 'Open Data Channel Type', 'Status']

Preprocessor created successfully!


In [19]:
# Step 7: Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Data split successfully!")

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Data split successfully!
Training features: (3120, 19)
Testing features: (781, 19)
Training target: (3120,)
Testing target: (781,)
